# Trigger-word dictionary audit: mislabeled entities

The supervised model is trained on labels produced by substring-matching a hand-built
trigger-word dictionary (`Classes_dic` in `pipeline/1-preprocess.ipynb`). `classify()`
assigns each event the **first** class (in dict order) whose **first** matching trigger
is a *substring* of the lowercased `notes`.

Raw substring matching + US/British spelling means some triggers fire in the wrong
context and poison the training labels. The original finding was `"labour conditions"`
filed under *animal welfare*. This notebook audits the whole dictionary and produces a
table of the resulting mislabeled entities, with counts attributed to the exact
deciding trigger.

In [1]:
import pandas as pd
import re

df = pd.read_csv('../../data/labeled.csv', low_memory=False)
df['nl'] = df['notes'].astype(str).str.lower()
print(f'{len(df)} trigger-word-labeled events')

214232 trigger-word-labeled events


## The dictionary under audit

Copied verbatim from `pipeline/1-preprocess.ipynb` so this notebook is self-contained
and reflects exactly what produced the labels.

In [2]:
Classes_dic = {
    "blm": ["black lives matter"],
    "lgbtq": ["lgb", "lesbian", "gay", "homosexual", "transsexual", "queer", "homophobia", "transphobia", "biphobia", "trans rights"],
    "women rights": ["women's rights", "feminism", "feminist", "against women", "women protested", "abortion", "sexual violence", "sexual assault", "sexual harassment", "sexual abuse"],
    "immigration": ["migrants", "immigration", "against migration", "deportation detention"],
    "unjust law enforcement": ["police brutality", "criminalize protests", "criminalize demonstrations", "police misconduct", "police repression"],
    "discrimination": ["discrimination", "racism"],
    "climate": ["climate change", "fossil fuels", "greenwashing", "climate agenda", "global warming"],
    "palestine-israel conflict": ["gaza", "palestine", "israel", "hamas", "palestinian"],
    "animal welfare": ["species extinction", "animal welfare", "animal rights", "labour conditions", "animal protection", "bullfighting", "animals locked", "wildlife", "cruel"],
    "farmers": ["farmers", "agriculture", "agricultural", "intensive farming"],
    "labor rights": ["labor agreement", " wages", "rights of workers", "labor rights", "higher salaries", "working conditions", "commission fees", "pension", "salary equalization", "unfairly dismissed"],
    "health care": ["healthcare", "hospital ", "hospitals", "nurseries", "emergency clinics", "emergency care"],
    "environment": ["environment", "pfas", "nitrogen", "planned felling", "biodiversity", "park project"],
    "public services": ["collapse of a concrete canopy", "canopy collapse", " bus ", "traffic accidents", "railway station", "bike lanes", "road connection", "public service", "pedestrianization", "child-safe intersections", "bike street", "play street", "reasonable mobility", "cycling conditions", "urban development", "free transport"],
    "ukraine-russia war": [" russia", "ukrain"],
    "housing": ["residential complex", "dignified housing", "evict"],
    "culture": ["tourism", "tourists", "cultural sector"],
    "policies & politics": ["social welfare", "social services", "social assistance", "economic justice", "economic sovereignty", "economic independence", "adoption of the euro", "euro adoption", "council's plan", "nightlife noise", "municipality", "regional government", "political criticism", "political opposition", "against the pm", "resignation of the president", "political rights", "political prisoners", "anti-eu", "pro-eu", "democratic", "referendums", "urgent elections", "distinct autonomy"],
    "pandemic": ["pandemic", "covid", "coronavirus"],
    "education": ["education", "teacher", "academic", "education", "professor", "university", "student loan"]
}

## Reconstruct the deciding trigger

We replicate `classify()` exactly, but record *which* `(class, trigger)` fired for each
event. This lets us attribute a mislabel to the precise trigger responsible, instead of
loosely pattern-matching. We sanity-check that the reconstruction reproduces the stored
labels.

In [3]:
def deciding(text):
    """First (class, trigger) whose trigger is a substring of `text` - mirrors classify()."""
    for cls, words in Classes_dic.items():
        for w in words:
            if w in text:
                return cls, w
    return 'unknown', None

fired = df['nl'].map(deciding)
df['fired_class'] = fired.map(lambda x: x[0])
df['fired_word'] = fired.map(lambda x: x[1])

agree = (df['fired_class'] == df['class']).mean()
print(f'reconstruction matches stored class on {agree:.4f} of rows')

reconstruction matches stored class on 1.0000 of rows


## Confirmed mislabels

Four triggers fire in the wrong context. For each we count only the events whose
**deciding** trigger is the buggy one *and* whose match is the bad kind (e.g. `pension`
only when it came from inside `suspension`, never a real pension protest).

In [4]:
has = lambda pat: df['nl'].str.contains(pat, regex=True)

# Each entry: trigger, the class it wrongly grabs into, a boolean mask of true mislabels,
# the mechanism, and the suggested fix.
CHECKS = [
    dict(trigger='pension', wrong_class='labor rights',
         mask=(df['fired_word'] == 'pension') & ~has(r'\bpension'),
         mechanism="substring of 'suspension'",
         fix=r"word-boundary match (\bpension\b)"),
    dict(trigger='labour conditions', wrong_class='animal welfare',
         mask=(df['fired_word'] == 'labour conditions'),
         mechanism="labour phrase filed under animal welfare",
         fix="move trigger to 'labor rights'"),
    dict(trigger='environment', wrong_class='environment',
         mask=(df['fired_word'] == 'environment')
              & has(r'working environment|business environment')
              & ~df['nl'].str.replace(r'working environment|business environment', '', regex=True).str.contains('environment'),
         mechanism="matches 'working/business environment' (labor protests)",
         fix="require 'environmental' / 'the environment'"),
    dict(trigger='nurseries', wrong_class='health care',
         mask=(df['fired_word'] == 'nurseries'),
         mechanism="nurseries = childcare/daycare, not healthcare",
         fix="drop trigger (typo for 'nurses'?)"),
]

rows = []
flagged_parts = []
for c in CHECKS:
    sub = df[c['mask']]
    rows.append({'trigger': c['trigger'], 'wrong_class': c['wrong_class'],
                 'mislabeled_rows': len(sub), 'mechanism': c['mechanism'], 'suggested_fix': c['fix']})
    part = sub[['event_id_cnty', 'class', 'notes']].copy()
    part.insert(1, 'bug_trigger', c['trigger'])
    flagged_parts.append(part)

summary = pd.DataFrame(rows).sort_values('mislabeled_rows', ascending=False).reset_index(drop=True)
flagged = pd.concat(flagged_parts, ignore_index=True)
print(f'Total confirmed mislabeled rows: {len(flagged)}')

Total confirmed mislabeled rows: 523


### Resulting table of mislabeled entities

In [5]:
summary

,trigger,wrong_class,mislabeled_rows,mechanism,suggested_fix
0,pension,labor rights,418,substring of 'suspension',word-boundary match (\bpension\b)
1,labour conditions,animal welfare,46,labour phrase filed under animal welfare,move trigger to 'labor rights'
2,environment,environment,41,matches 'working/business environment' (labor ...,require 'environmental' / 'the environment'
3,nurseries,health care,18,"nurseries = childcare/daycare, not healthcare",drop trigger (typo for 'nurses'?)


A few example events behind each trigger, to make the failure concrete.

In [6]:
for trig in summary['trigger']:
    ex = flagged[flagged['bug_trigger'] == trig].head(2)
    print(f"\n=== {trig} (-> labeled '{ex['class'].iloc[0]}') ===")
    for t in ex['notes']:
        print("  -", str(t)[:150])


=== pension (-> labeled 'labor rights') ===
  - On 25 July 2018, 2018: Citizens protested in Limassol, calling for the suspension of construction works on a petrol station on Dimokratias Street due 
  - On 6 September 2018, 2018 in Podgorica, Montenegro the Trade Union of Administration and Judiciary of Montenegro organized a protest over suspension o

=== labour conditions (-> labeled 'animal welfare') ===
  - On 10 January 2020, members of school staff gathered outside Illostrin National School in Letterkenny, Co. Donegal before marching to the office Educa
  - On 10 January 2020, teachers and union representatives staged a protest outside the North Dublin National School Project in Ballymun, Dublin before ma

=== environment (-> labeled 'environment') ===
  - On 26 February 2020, more than 150 people, supported by the Swedish Municipal Workers' Union, the Union for Professionals, Vision and Saco, protested 
  - On 22 January 2021, striking employees of Orezza, as well as STC delega

The full list of flagged events is written to `mislabeled_rows.csv`.

In [7]:
flagged.to_csv('mislabeled_rows.csv', index=False)
print(f'wrote {len(flagged)} rows to mislabeled_rows.csv')
flagged.head(10)

wrote 523 rows to mislabeled_rows.csv


,event_id_cnty,bug_trigger,class,notes
0,CYP48,pension,labor rights,"On 25 July 2018, 2018: Citizens protested in L..."
1,MNE58,pension,labor rights,"On 6 September 2018, 2018 in Podgorica, Monten..."
2,ROU475,pension,labor rights,"On 18 June 1500 people, mainly Hungarians, pro..."
3,ROU1106,pension,labor rights,"On 3 November 2019, tens of people protested i..."
4,MKD272,pension,labor rights,"On 23 November 2019, citizens held a protest o..."
5,MKD294,pension,labor rights,"On 27 December 2019, former members of the Ind..."
6,GBR462,pension,labor rights,"On 3 March 2020, around 100 people, including ..."
7,FRA1584,pension,labor rights,"On 15 March 2020, around 100 prisoners demonst..."
8,ESP454,pension,labor rights,"On 16 March 2020, around 5000 workers of Merce..."
9,FRA1593,pension,labor rights,"On 17 March 2020, prisoners demonstrated in Pa..."


## Recall gap: US vs British spelling

The labor-rights triggers are US-spelled (`labor`), but the ACLED notes are British
(`labour`). So events that explicitly say *"labour rights"* mostly never reach the
labor-rights class - the same root cause as `labour conditions`, but causing misses
rather than wrong grabs.

In [8]:
mask_labour = has(r'\blabour rights|\blabour agreement')
caught = (df[mask_labour]['class'] == 'labor rights').mean()
print(f"events saying 'labour rights/agreement': {mask_labour.sum()}")
print(f"  of which actually labeled 'labor rights': {caught:.0%}")
print(f"  -> {1-caught:.0%} slip through due to US/British spelling")

events saying 'labour rights/agreement': 93
  of which actually labeled 'labor rights': 28%
  -> 72% slip through due to US/British spelling
